# 02 Generation
**Purpose of this notebook:** construct the synthetic half of the paired dataset. For every human review prepared in 01, one LLM review is generated from the same movie title, release year and sentiment direction.

- The 100 group-A human reviews are paired with **LLM A = gpt-4o-mini** (OpenAI)
- The 100 group-B human reviews are paired with **LLM B = claude-haiku-4-5** (Anthropic)

Both models receive an identical prompt template and identical decoding parameters, so that the model identity is the only variable distinguishing the two synthetic subsets.

**Input:** `human_reviews.csv` (from 01)
**Output:** `llm_reviews.csv` (200 synthetic reviews), `all_texts.csv` (the full 400-text dataset)

Generation is resumable: rerunning the notebook after an interruption skips the items already produced.

## Step 1: Install packages and provide API keys
Keys are entered interactively and held in environment variables only; they are never written to any output file, so no credentials can reach the repository.

In [ ]:
!pip install -q openai anthropic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

## Step 2: Load the human reviews
The file produced by 01 supplies the movie metadata, the group assignment and the ready-made generation prompt for each of the 200 human reviews.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

human = pd.read_csv("human_reviews.csv")
print(len(human), "human reviews")
print(human["llm_group"].value_counts())
assert "prompt" in human.columns, "缺少 prompt 列，请重新运行 01"
human.head(2)

## Step 3: Configuration
All generation parameters are declared in one place so that the settings reported in the Methodology section of the poster correspond exactly to the code that produced the data.

In [ ]:
MODEL_A = "gpt-4o-mini"                    # LLM A (OpenAI)
MODEL_B = "claude-haiku-4-5-20251001"      # LLM B (Anthropic)
TEMPERATURE = 0.7
MAX_TOKENS = 400
MIN_WORDS, MAX_WORDS = 100, 150
MAX_ATTEMPTS = 3                           # 字数不合格时最多重试次数
OUT_FILE = "llm_reviews.csv"


## Step 4: Post-processing and length measurement
Language models frequently return a headline, Markdown emphasis or surrounding quotation marks. Such formatting artefacts would inflate the punctuation features and distort sentence segmentation, and are therefore stripped before any measurement. Note that this cleaning applies to the generated texts only; the human reviews are left as published.

In [ ]:
import re

def clean_output(text):
    text = text.strip()
    text = re.sub(r"^```[a-z]*\n|```$", "", text)          # 代码块
    text = re.sub(r"[*#_`]", "", text)                      # Markdown 符号
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    # 删除单独的标题行（很短且不以句末标点结尾）
    if len(lines) > 1 and len(lines[0].split()) < 12 and not lines[0].endswith((".", "!", "?")):
        lines = lines[1:]
    text = " ".join(lines)
    text = text.strip().strip('"').strip("'")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text):
    return len(text.split())

# 测试
demo = "**Review: Titanic**\n\nThis film is *wonderful*.  Truly great."
print(repr(clean_output(demo)), word_count(clean_output(demo)))

## Step 5: Model interfaces
One function per provider, both called with the same prompt, temperature and token limit. A single test generation per model verifies that both APIs respond correctly before the batch run starts.

In [ ]:
from openai import OpenAI
import anthropic

openai_client = OpenAI()
anthropic_client = anthropic.Anthropic()

def generate_openai(prompt):
    resp = openai_client.chat.completions.create(
        model=MODEL_A,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return resp.choices[0].message.content

def generate_anthropic(prompt):
    resp = anthropic_client.messages.create(
        model=MODEL_B,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return resp.content[0].text

def generate(group, prompt):
    return generate_openai(prompt) if group == "A" else generate_anthropic(prompt)

# 先测一条，确认两个 key 都能用
test_prompt = human["prompt"].iloc[0]
print("PROMPT:", test_prompt)
print("\nOpenAI  ->", clean_output(generate_openai(test_prompt))[:120], "...")
print("\nAnthropic ->", clean_output(generate_anthropic(test_prompt))[:120], "...")

## Step 6: Batch generation
Each record stores the model name, temperature, number of attempts, generation timestamp and the unprocessed model output, which together make the dataset reproducible.

A generation whose length falls outside the 100–150 word range is regenerated, up to three attempts. If no attempt satisfies the constraint, the final attempt is retained and flagged, so that the number of such cases can be reported as a limitation rather than silently discarded.

In [ ]:
import time
from datetime import datetime, timezone

results = []
if os.path.exists(OUT_FILE):
    results = pd.read_csv(OUT_FILE).to_dict("records")
    print("已有", len(results), "条，继续生成剩下的")
done = {r["review_id"] for r in results}

for i, row in human.iterrows():
    if row["review_id"] in done:
        continue
    group = row["llm_group"]
    model = MODEL_A if group == "A" else MODEL_B
    raw, text, wc = None, None, 0
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            raw = generate(group, row["prompt"])
        except Exception as e:
            print("  错误，20 秒后重试:", e)
            time.sleep(20)
            continue
        text = clean_output(raw)
        wc = word_count(text)
        if MIN_WORDS <= wc <= MAX_WORDS:
            break
    results.append({
        "review_id": row["review_id"],          # 与人类影评配对的 ID
        "llm_group": group,
        "source": f"llm_{group}",
        "model": model,
        "temperature": TEMPERATURE,
        "tconst": row["tconst"],
        "primaryTitle": row["primaryTitle"],
        "label": row["label"],
        "prompt": row["prompt"],
        "text": text,
        "word_count": wc,
        "in_range": MIN_WORDS <= wc <= MAX_WORDS,
        "attempts": attempt,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "raw_output": raw,
    })
    if len(results) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_FILE, index=False)
        print(len(results), "/", len(human), "done")
    time.sleep(0.5)

llm = pd.DataFrame(results)
llm.to_csv(OUT_FILE, index=False)
print("完成:", len(llm))

## Step 7: Verification of the generated texts
Checks the number of texts per model, the word-count distribution, the number of length violations, the distribution of retry counts, residual formatting artefacts, duplicate outputs, and whether every human review received a counterpart.

In [ ]:
print("Total:", len(llm))
print(llm["source"].value_counts())
print()
print(llm.groupby("source")["word_count"].describe()[["count", "mean", "min", "max"]])
print()
print("字数不合格:", (~llm["in_range"]).sum())
print("重试次数分布:")
print(llm["attempts"].value_counts().sort_index())
print()
print("残留 Markdown:", llm["text"].str.contains(r"[*#]", regex=True).sum())
print("重复文本:", llm["text"].duplicated().sum())
print("配对完整:", set(llm["review_id"]) == set(human["review_id"]))

## Step 8: Assembly of the final dataset
`all_texts.csv` contains all 400 texts and serves as the input to 03.

- `source`: human / llm_A / llm_B
- `is_llm`: 0 = human, 1 = synthetic (classification label)
- `tconst`: movie ID, used as the grouping variable in cross-validation so that a human review and its synthetic counterpart never fall into different folds
- `llm_group`: A / B, defining the training and test sets of the cross-LLM experiment

In [ ]:
cols = ["review_id", "tconst", "primaryTitle", "label", "llm_group", "text", "word_count"]

h = human[cols].copy()
h["source"] = "human"
h["model"] = "human"

l = llm[cols + ["source", "model"]].copy()

all_texts = pd.concat([h, l], ignore_index=True)
all_texts["is_llm"] = (all_texts["source"] != "human").astype(int)
all_texts["text_id"] = all_texts["source"] + "_" + all_texts["review_id"]

all_texts.to_csv("all_texts.csv", index=False)

print(len(all_texts), "texts")
print(pd.crosstab(all_texts["source"], all_texts["llm_group"]))
print()
print("每部电影的文本数（应全部为 2）:", all_texts["tconst"].value_counts().unique())

In [ ]:
files.download("llm_reviews.csv")
files.download("all_texts.csv")